# 04 — Hyperparameter Tuning (Phase 4) & Threshold Optimization (Phase 5)

Two phases combined into one notebook because threshold selection naturally
follows the tuned model. Both stages use only `X_train` (further split
internally) — **the test set remains untouched.**

In [ ]:
from src.config import get_default_config
from src.data.load import load_and_validate_data
from src.data.split import train_test_split_data
from src.models.pipeline import build_model_pipeline
from src.models.tuning import run_randomized_search
from src.threshold.optimizer import select_threshold, apply_threshold

config = get_default_config()
config.data.target_column = "<set explicitly — dataset-specific>"
config.model.model_type = "xgboost"

In [ ]:
df, schema = load_and_validate_data(config.data)
config.preprocessing.numerical_features = schema.numerical_features
config.preprocessing.categorical_features = schema.categorical_features

X_train, X_test, y_train, y_test = train_test_split_data(
    df, config.data.target_column, config.split
)

## Phase 4 — RandomizedSearchCV over XGBoost hyperparameters

Search space is regularization-focused (`max_depth`, `min_child_weight`,
`subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`, `learning_rate`,
`n_estimators`). Native early stopping via `eval_set` was deliberately
**not** wired into the CV loop — correctly preprocessing a fold-specific
eval set inside a `Pipeline` risks leakage or preprocessing mismatches, so
generalization gap is instead monitored post-hoc (see below).

In [ ]:
base_pipeline = build_model_pipeline(config.model, config.preprocessing)

param_distributions = {
    "classifier__max_depth": [3, 4, 5, 6, 7],
    "classifier__min_child_weight": [1, 3, 5, 7],
    "classifier__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "classifier__reg_alpha": [0, 0.01, 0.1, 1.0],
    "classifier__reg_lambda": [0.5, 1.0, 1.5, 2.0],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__n_estimators": [100, 200, 300, 500],
}

search_result = run_randomized_search(
    base_pipeline, param_distributions, X_train, y_train, config.tuning, config.cv
)

best_pipeline = search_result.best_estimator_
print("Best params:", search_result.best_params_)
print("Best CV score:", search_result.best_score_)

In [ ]:
import pandas as pd

cv_results = pd.DataFrame(search_result.cv_results_)
cv_results.sort_values("rank_test_score").head(10)[
    [c for c in cv_results.columns if c.startswith("param_") or "mean_test_score" in c]
]

### Train/validation gap check

A gap above `config.tuning.train_val_gap_warning_threshold` (default 0.10)
is logged as a warning by `run_randomized_search`, not enforced/auto-corrected.

In [ ]:
print("train_score:", search_result.train_score_)
print("cv_score:", search_result.best_score_)
print("gap:", search_result.train_score_ - search_result.best_score_)

## Phase 5 — Threshold optimization

Uses a holdout split carved out of `X_train` (never the test set). The tuned
pipeline is refit on the remaining training data and evaluated on this
holdout purely to select a decision threshold.

In [ ]:
from src.data.split import train_test_split_data as _split  # reused helper

X_fit, X_holdout, y_fit, y_holdout = _split(
    pd.concat([X_train, y_train], axis=1),
    config.data.target_column,
    config.split,  # test_size here plays the role of the threshold-holdout fraction
)

best_pipeline.fit(X_fit, y_fit)
y_holdout_proba = best_pipeline.predict_proba(X_holdout)[:, 1]

In [ ]:
config.threshold.strategy = "maximize_f1"
threshold_result = select_threshold(y_holdout, y_holdout_proba, config.threshold)
threshold_result

In [ ]:
# Compare against an alternative strategy for inspection (not selection)
config.threshold.strategy = "youden_j"
alt_threshold_result = select_threshold(y_holdout, y_holdout_proba, config.threshold)
alt_threshold_result

### cost_based strategy demo (requires explicit costs — no default policy)

In [ ]:
config.threshold.strategy = "cost_based"
config.threshold.cost_fp = 1.0   # caller-supplied — illustrative only
config.threshold.cost_fn = 5.0   # caller-supplied — illustrative only

cost_threshold_result = select_threshold(y_holdout, y_holdout_proba, config.threshold)
cost_threshold_result

In [ ]:
preds = apply_threshold(y_holdout_proba, threshold_result.threshold)
preds[:20]

## Notes

- Chosen threshold strategy and value for downstream phases: `<fill in>`
- `cost_fp`/`cost_fn` values above are illustrative placeholders, not a
  business policy embedded in `src/` — real values must come from the
  caller/business stakeholder at run time.
- `y_test` remains completely unused in this notebook.